# 00 — Data smoke-test (Phase 0)

**Purpose: prove the whole pipeline runs end-to-end on one session before we touch the science.**

The HMM at the end is *not* a result. It is a canary. If it fits, then data access,
trial alignment, binning, and model fitting all work, and Phase 1 can start.

Steps:
1. Connect to public **openalyx** (no account needed).
2. Pull the BWM metadata tables; count good units per region.
3. Find sessions where our candidate-axis regions were recorded **simultaneously**.
4. *(gated)* Download one insertion's spikes + the session trials table.
5. Build a `(trials x time x neurons)` tensor aligned to stimulus onset.
6. Fit a Poisson HMM and relate the state sequence to reaction time.

Steps 1-3 are **metadata only** and cost nothing. Step 4 downloads ~440 MB.

Run with the `neurodatarehack` conda env.

In [ ]:
import sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# make `src` importable when running from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml

from src.data.access import get_one, sessions_table, units_table, region_cooccurrence, load_units, load_trials
from src.data.build_matrices import bin_spikes_by_event, stack_for_hmmlearn, drop_silent_units

PARAMS = yaml.safe_load((ROOT / "config" / "params.yaml").read_text())
AREAS = yaml.safe_load((ROOT / "config" / "areas.yaml").read_text())
AXIS = AREAS["candidate_axis"]

rng = np.random.default_rng(PARAMS["random_seed"])
print("candidate axis:", AXIS)

## 1. Connect to openalyx

`get_one()` uses ONE-api's built-in public credentials and pins the aggregate-table
tag so `bwm_units` reproduces the paper's frozen unit set. No registration needed.

In [ ]:
one = get_one(cache_dir=PARAMS["data"]["one_cache_dir"])
print("mode:      ", one.mode)
print("cache_dir: ", one.cache_dir)

## 2. Metadata tables and good-unit yields

`units_table` asserts a hash of the unit UUIDs against the `2023_12_bwm_release`
freeze, so these are exactly the units the paper analysed.

Expect **699 insertions / 459 sessions**, and **62,990 units over 210 Beryl regions**.

In [ ]:
bwm_df = sessions_table(one)
unit_df = units_table(one)

print(f"insertions: {bwm_df.pid.nunique()}   sessions: {bwm_df.eid.nunique()}")
print(f"good units: {len(unit_df)}   Beryl regions: {unit_df.Beryl.nunique()}")

yields = pd.DataFrame({
    "good_units": unit_df[unit_df.Beryl.isin(AXIS)].Beryl.value_counts().reindex(AXIS),
    "n_sessions": unit_df[unit_df.Beryl.isin(AXIS)].groupby("Beryl", observed=True).eid.nunique().reindex(AXIS),
})
yields

## 3. Which axis regions co-occur in one session?

Single-trial methods (HMM, rSLDS, geometry, flow) need **simultaneously recorded**
populations. Pooling across sessions into a pseudopopulation destroys the trial-by-trial
correlations these models exist to capture. So the unit of selection is the *session*.

In [ ]:
sel = AREAS["selection"]
co = region_cooccurrence(
    unit_df, AXIS,
    min_units_per_region=sel["min_units_per_region"],
    min_regions_per_session=sel["min_regions_per_session"],
)
print(f"{len(co)} sessions have >= {sel['min_regions_per_session']} axis regions "
      f"with >= {sel['min_units_per_region']} good units each\n")
for _, r in co.head(6).iterrows():
    print(f"  {r.eid}  n_reg={r.n_regions}  " + ", ".join(f"{a}:{b}" for a, b in zip(r.regions, r.units)))

In [ ]:
# Pick the richest session, and within it the insertion carrying the most units
# of a single target region. That region is our smoke-test population.
best = co.iloc[0]
EID = best.eid

cand = (unit_df[(unit_df.eid == EID) & (unit_df.Beryl.isin(AXIS))]
        .groupby(["pid", "Beryl"], observed=True).size()
        .reset_index(name="n").sort_values("n", ascending=False))
display(cand)

PID = cand.iloc[0]["pid"]
REGION = cand.iloc[0]["Beryl"]
N_EXPECTED = int(cand.iloc[0]["n"])
print(f"\nsmoke-test target -> eid={EID}\n                    pid={PID}\n"
      f"                    region={REGION} ({N_EXPECTED} good units)")

## 4. Download (gated)

> **This is the only cell that downloads neural data.**
> With `qc=1.0`, `load_good_units` fetches `passingSpikes.table.pqt` (a good-units-only
> spike table) plus small `clusters.*`/`channels.*` files — **not** the raw ephys.
> Roughly **150–280 MB per insertion**. Cached afterwards, so re-running is free.

Set `DOWNLOAD = True` to proceed.

In [ ]:
DOWNLOAD = True  # <-- flip to True to fetch data

if not DOWNLOAD:
    raise RuntimeError("Set DOWNLOAD = True to download ~150-280 MB for this insertion.")

spikes, clusters = load_units(one, PID, qc=PARAMS["data"]["min_qc"])
trials, mask = load_trials(
    one, EID,
    min_rt=PARAMS["trials"]["min_rt_s"],
    max_rt=PARAMS["trials"]["max_rt_s"],
)
print(f"spikes: {spikes['times'].size:,} from {clusters.shape[0]} good units")
print(f"trials: {mask.sum()} usable of {len(trials)} total")

### Select the units belonging to our Beryl region

`clusters['acronym']` holds the *fine-grained* Allen acronym (e.g. `SCig`, `SCiw`,
`VISp2/3`). Beryl is a coarser rollup (`SCm`, `VISp`). We must map before filtering,
otherwise `acronym == 'SCm'` silently matches nothing.

In [ ]:
from iblatlas.regions import BrainRegions

br = BrainRegions()
clusters = clusters.copy()
clusters["Beryl"] = br.id2acronym(clusters["atlas_id"].values, mapping="Beryl")

unit_idx = np.flatnonzero((clusters["Beryl"] == REGION).values)
print(f"{len(unit_idx)} units in {REGION} (metadata said {N_EXPECTED})")
print(clusters["Beryl"].value_counts().head(8).to_string())

## 5. Build the (trials x time x neurons) tensor

Aligned to `stimOn_times`. The trial mask is applied first — it enforces the paper's
own criteria (reaction time in range, no NaN in key events).

In [ ]:
B = PARAMS["binning"]
good_trials = trials[mask]
events = good_trials[B["align_event"]].values

# keep only spikes from our region's units, and re-index those units to 0..n-1
keep_spike = np.isin(spikes["clusters"], unit_idx)
remap = -np.ones(clusters.shape[0], dtype=np.int64)
remap[unit_idx] = np.arange(unit_idx.size)

counts, bin_centers = bin_spikes_by_event(
    spike_times=spikes["times"][keep_spike],
    spike_clusters=remap[spikes["clusters"][keep_spike]],
    n_clusters=unit_idx.size,
    event_times=events,
    pre_s=B["pre_s"], post_s=B["post_s"], bin_size_s=B["bin_size_s"],
)
counts, kept = drop_silent_units(counts, min_total_spikes=1)
print(f"tensor: {counts.shape}  (trials x bins x neurons), dtype={counts.dtype}")
print(f"window: {bin_centers[0]:+.3f} .. {bin_centers[-1]:+.3f} s around {B['align_event']}")
print(f"mean rate: {counts.sum() / (counts.shape[0]*counts.shape[1]*B['bin_size_s']*counts.shape[2]):.2f} sp/s")

In [ ]:
# Sanity: population PSTH should show a visual response after stimulus onset.
psth = counts.mean(axis=(0, 2)) / B["bin_size_s"]
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(bin_centers, psth, lw=1.5)
ax.axvline(0, color="k", ls="--", lw=1, label="stimulus onset")
ax.set(xlabel="time from stimOn (s)", ylabel="rate (sp/s)", title=f"{REGION} population PSTH")
ax.legend(frameon=False); fig.tight_layout()

## 6. Fit a Poisson HMM (the canary)

Poisson emissions are the right likelihood for **raw spike counts** — so we deliberately
feed unsmoothed integers. `lengths` tells `hmmlearn` that trials are independent
sequences, so it never learns a transition across a trial boundary.

`n_states` here is an arbitrary Phase-0 value. Choosing K by cross-validated
log-likelihood is a **Phase 2** job — do not read anything into this fit.

In [ ]:
from hmmlearn.hmm import PoissonHMM

K = PARAMS["hmm"]["n_states"]
X, lengths = stack_for_hmmlearn(counts)

model = PoissonHMM(n_components=K, n_iter=PARAMS["hmm"]["n_iter"], random_state=PARAMS["random_seed"])
model.fit(X, lengths)

print(f"converged: {model.monitor_.converged}   iters: {model.monitor_.iter}")
print(f"log-likelihood: {model.score(X, lengths):,.1f}")

states = model.predict(X, lengths).reshape(counts.shape[0], counts.shape[1])
print(f"state sequence: {states.shape} (trials x bins)")

In [ ]:
# Do the inferred states carry anything behavioural?
# Sort trials by reaction time; if states track the decision, structure should tilt.
rt = (good_trials["firstMovement_times"] - good_trials["stimOn_times"]).values
order = np.argsort(rt)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), gridspec_kw={"width_ratios": [2, 1]})

im = axes[0].imshow(states[order], aspect="auto", interpolation="nearest",
                    extent=[bin_centers[0], bin_centers[-1], len(order), 0], cmap="viridis")
axes[0].plot(rt[order], np.arange(len(order)) + 0.5, color="r", lw=1.2, label="first movement")
axes[0].axvline(0, color="w", ls="--", lw=1)
axes[0].set(xlabel="time from stimOn (s)", ylabel="trial (sorted by RT)",
            title=f"{REGION}: inferred HMM state, K={K}")
axes[0].legend(frameon=False, loc="lower right")
fig.colorbar(im, ax=axes[0], label="state")

occupancy = np.stack([(states == k).mean(axis=0) for k in range(K)])
for k in range(K):
    axes[1].plot(bin_centers, occupancy[k], label=f"state {k}")
axes[1].axvline(0, color="k", ls="--", lw=1)
axes[1].set(xlabel="time from stimOn (s)", ylabel="P(state)", title="state occupancy")
axes[1].legend(frameon=False, fontsize=8)
fig.tight_layout()

## What passing means

If every cell above ran, then the following are all verified on real data:
openalyx access → frozen unit set → simultaneous-region selection → spike download →
trial masking → stimulus-aligned binning → Poisson HMM fit → behavioural readout.

That is Phase 0 complete. **Nothing here is a scientific result.**

Next — **Phase 1 (area selection):** widen step 3 across all 43 qualifying sessions,
decide the final region set, and freeze it into `config/areas.yaml` under `selected`.

## 7. One worked example per area — PSTH with an error band

Steps 4–6 used a *single* probe, so they only saw **SCm**. To show all three regions of the
frozen axis — **VISp, SCm, MRN** — we now pool **both probes** of this session (recorded
simultaneously, sharing the trial clock) through the project's **unified preprocessing**
(`src/data/preprocess.py`): `stimOn`-aligned, `pre 0.4 / post 0.8 s`, **40 ms** bins, units ≥ 1 Hz.
That is the exact tensor every downstream model consumes, so these panels double as a check that
the shared pipeline runs.

The **shaded band is ±1 SEM across trials** of the population firing rate — on each trial the
population rate is the mean over that region's units, and the band is (std over trials)/√(n trials).
It is spelled out on every panel with that area's own trial and unit counts, because an "error
band" says nothing unless you state *what it is computed over*. (SEM across **trials** answers
"how reproducible is the mean response from trial to trial"; a band across **units** would answer a
different question — how much units differ from each other.)


In [ ]:
# Sections 7-8 use BOTH probes (so all three regions appear) and the UNIFIED pipeline.
from src.data.preprocess import (
    PreprocessConfig, load_session, align_times, usable_mask, preprocess_session,
)
from src.viz.style import set_style, region_color, stroke, savefig, OKABE_ITO

set_style()
REGIONS_3 = ("VISp", "SCm", "MRN")                 # the frozen visual->motor axis
EID = "8a3a0197-b40a-449f-be55-c00b23253bbf"       # smoke-test session (UCLA034), pinned

# frozen preprocessing: stimOn-aligned, pre 0.4 / post 0.8 s, 40 ms bins, units >= 1 Hz
cfg = PreprocessConfig(align_event="stimOn_times", pre_s=0.4, post_s=0.8,
                       bin_size_s=0.04, min_rate_hz=1.0)

trials3, mask3, probe_loads = load_session(one, EID, cfg=cfg)     # loads BOTH probes

# trial-level covariates, kept in register with axis 0 of the tensors (same order as align_times)
um = usable_mask(trials3, mask3, cfg.align_event)  # finite stimOn -> all True for this alignment
tt = trials3[mask3][um]
cL = tt["contrastLeft"].to_numpy(float)
cR = tt["contrastRight"].to_numpy(float)
choice = tt["choice"].to_numpy(float)                        # IBL: -1 / +1 / 0 (no-go)
signed_contrast = np.nan_to_num(cR, nan=0.0) - np.nan_to_num(cL, nan=0.0)   # + = RIGHT stim
side = np.sign(signed_contrast)                              # +1 right, -1 left, 0 at 0% contrast
abs_contrast = np.abs(signed_contrast)

# raw-count tensors, one per region, all sharing the (trials, bins) axes
data3 = preprocess_session(probe_loads, align_times(trials3, mask3, cfg.align_event), cfg)
bc = data3["bin_centers"]
bin_s = cfg.bin_size_s
print("\ntrials:", len(tt), " | contrast levels (%):", (np.unique(abs_contrast) * 100).round(2))


In [ ]:
# PSTH per area, each with a +/-1 SEM-ACROSS-TRIALS band, annotated per panel.
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4), sharex=True)
for ax, r in zip(axes, REGIONS_3):
    counts = data3["regions"][r]["counts"]              # (trials, bins, units), raw integer counts
    n_tr, n_bin, n_u = counts.shape
    pop_rate = counts.mean(axis=2) / bin_s              # per-trial population rate (trials, bins)
    psth = pop_rate.mean(axis=0)                        # mean over trials
    sem = pop_rate.std(axis=0, ddof=1) / np.sqrt(n_tr)  # <-- SEM ACROSS TRIALS (std / sqrt n_trials)
    col = region_color(r)
    ax.fill_between(bc, psth - sem, psth + sem, color=col, alpha=0.30, linewidth=0)
    ax.plot(bc, psth, color=col, lw=1.8, path_effects=stroke())
    ax.axvline(0, color="k", ls="--", lw=1)
    ax.set_title(r, fontweight="bold")
    ax.set_xlabel("time from stimOn (s)")
    # write, per area, WHAT the band is computed over -- an error band is meaningless otherwise
    ax.text(0.03, 0.97,
            f"shaded = ±1 SEM\nacross {n_tr} trials\n(pop. rate = mean\nover {n_u} units)",
            transform=ax.transAxes, va="top", ha="left", fontsize=8,
            bbox=dict(boxstyle="round", fc="white", ec=col, alpha=0.9))
axes[0].set_ylabel("population rate (sp/s)")
fig.suptitle(f"Population PSTH per area · session {EID[:8]} · aligned to stimulus onset",
             y=1.04, fontweight="bold")
fig.tight_layout()
print("saved:", savefig(fig, "smoketest", "psth_per_area"))


## 8. Decoding **choice** at different contrast ("coherence") — and why the accuracy can look "wrong"

The worry *"decoding accuracy doesn't move with coherence the way it should"* is the most common
trap in this task, and it comes from decoding **the wrong variable**.

**What "coherence" is here.** IBL does not use random-dot coherence; it uses a Gabor of varying
**contrast** ∈ {0, 6.25, 12.5, 25, 100} %. Contrast is the rodent analogue of coherence — the
strength of the *sensory evidence*. On each trial the Gabor appears on the **left or right** and
the mouse turns a wheel to bring it to centre. The **choice** is the wheel-turn direction.

**Two different things you can decode, and they depend on contrast oppositely:**

| decode | what it is | expected dependence on contrast |
|---|---|---|
| **stimulus side** | which side the Gabor was on | **rises** with contrast — it *is* the sensory signal |
| **choice** | which way the mouse actually turned | **need not rise** — the mouse moves on every trial |

The key fact: **the mouse commits to a movement on every single trial, including at 0 % contrast**
(there it guesses, biased by the block prior `probabilityLeft`). Motor-related areas (SCm, MRN)
encode the *executed action*, so **choice is decodable at all contrasts — even when there is no
stimulus to see.** If you decode *choice* and expect its accuracy to climb with contrast like a
sensory curve, it will not, and nothing is broken. The variable whose decoding climbs with
contrast is the **stimulus**, not the choice.

Two things sharpen the confusion, and the code below handles both:

1. **At high contrast, choice ≈ stimulus** (the mouse is ~always correct), so the two decoders
   *converge* — which can masquerade as "choice tracks contrast". At **low** contrast they
   *diverge*: stimulus decoding falls to chance while choice decoding stays high. Watch the gap.
2. **Class imbalance inflates plain accuracy.** At high contrast one choice dominates, so a decoder
   that always guesses the majority scores ~0.8 for free. We therefore report **balanced accuracy**
   under **stratified** cross-validation; chance = 0.5.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score

# Empirically in this dataset, choice == -1 means the mouse CHOSE THE RIGHT stimulus:
# at |contrast| >= 50% (mouse ~always correct) choice==-1 co-occurs with side==+1.
chose_right = (choice == -1).astype(int)
go = choice != 0                       # drop the 3 no-go trials for the choice decoder


def window_features(counts, t0, t1):
    """Mean spike count per neuron over the window [t0, t1) s -> (trials, neurons)."""
    sel = (bc >= t0) & (bc < t1)
    return counts[:, sel, :].mean(axis=1)


def decode(X, y, n_splits=5, seed=0):
    """Balanced accuracy of an L2-logistic decoder under stratified CV. Returns (mean, sd).

    Balanced accuracy (not plain accuracy) + class_weight='balanced' so a decoder cannot
    win by always guessing the majority class -- the trap at high contrast where one choice
    dominates. Returns NaN if a class is too small to stratify.
    """
    y = np.asarray(y)
    classes, n = np.unique(y, return_counts=True)
    if len(classes) < 2 or n.min() < n_splits:
        return np.nan, np.nan
    clf = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=2000, C=0.5, class_weight="balanced"),
    )
    cv = StratifiedKFold(n_splits, shuffle=True, random_state=PARAMS["random_seed"])
    s = cross_val_score(clf, X, y, cv=cv, scoring="balanced_accuracy")
    return s.mean(), s.std()


STIM_WIN = (0.05, 0.20)     # early, sensory-dominated (before median RT ~0.17 s)
CHOICE_WIN = (0.0, 0.40)    # broad, spans movement onset
levels = np.unique(abs_contrast)
lev_pct = levels * 100

# ---- figure: psychometric (behaviour) + decoding-vs-contrast per area ----
fig = plt.figure(figsize=(13, 6.6))
gs = fig.add_gridspec(2, 3, height_ratios=[1, 1.15], hspace=0.45, wspace=0.28)

# psychometric: P(choose right) vs SIGNED contrast -- the behaviour the neural curves explain
axp = fig.add_subplot(gs[0, 0])
sc_levels = np.unique(signed_contrast)
p_right = np.array([np.mean(chose_right[signed_contrast == s]) for s in sc_levels])
axp.plot(sc_levels * 100, p_right, "-o", color=OKABE_ITO[4], lw=1.8, path_effects=stroke())
axp.axhline(0.5, color="k", ls=":", lw=1); axp.axvline(0, color="k", ls=":", lw=1)
axp.set(xlabel="signed contrast (%)   [+ = right stim]", ylabel="P(choose right)",
        title="Behaviour: psychometric", ylim=(-0.03, 1.03))

axt = fig.add_subplot(gs[0, 1:]); axt.axis("off")
axt.text(0.0, 1.0,
    "Reading the decoders\n"
    "- STIMULUS-SIDE (blue): which side the Gabor was on; |contrast|>0 trials only.\n"
    "   The SENSORY signal -- it should, and does, rise with contrast ('coherence').\n"
    "- CHOICE (orange): which way the mouse turned the wheel.\n"
    "   The mouse acts on EVERY trial (even 0%), so choice is decodable at all\n"
    "   contrasts and need NOT track contrast -- motor areas encode the action.\n"
    "- Balanced accuracy, 5-fold stratified CV; chance = 0.5 (dotted).\n"
    "   Bars = +/-1 SD across folds. Windows: stim [50,200] ms, choice [0,400] ms.",
    va="top", ha="left", fontsize=9, family="monospace")

for j, r in enumerate(REGIONS_3):
    ax = fig.add_subplot(gs[1, j])
    counts = data3["regions"][r]["counts"]
    Xs = window_features(counts, *STIM_WIN)
    Xc = window_features(counts, *CHOICE_WIN)
    sm, se, cm, ce = [], [], [], []
    for L in levels:
        m = abs_contrast == L
        a, e = decode(Xs[m & (side != 0)], side[m & (side != 0)]) if L > 0 else (np.nan, np.nan)
        sm.append(a); se.append(e)
        a2, e2 = decode(Xc[m & go], chose_right[m & go])
        cm.append(a2); ce.append(e2)
    ax.axhline(0.5, color="k", ls=":", lw=1)
    ax.errorbar(lev_pct, sm, yerr=se, fmt="-s", color=OKABE_ITO[4], lw=1.6,
                capsize=3, path_effects=stroke(), label="stimulus side")
    ax.errorbar(lev_pct, cm, yerr=ce, fmt="-o", color=OKABE_ITO[0], lw=1.6,
                capsize=3, path_effects=stroke(), label="choice")
    ax.set_xscale("symlog", linthresh=5)
    ax.set_xticks(lev_pct); ax.set_xticklabels([f"{v:g}" for v in lev_pct], fontsize=8)
    ax.set(title=f"{r}  ({counts.shape[2]} units)", xlabel="|contrast| (%)", ylim=(0.30, 1.0))
    if j == 0:
        ax.set_ylabel("balanced accuracy")
        ax.legend(frameon=False, fontsize=8, loc="upper left")

fig.suptitle(f"Decoding choice vs stimulus at different contrast ('coherence') · session {EID[:8]}",
             y=0.98, fontweight="bold")
print("saved:", savefig(fig, "smoketest", "decoding_vs_contrast"))


### What the panels say (session 8a3a0197, 287 usable go-trials)

- **Stimulus-side decoding climbs with contrast**, exactly as the "coherence" intuition predicts:
  SCm **0.36 → 0.93** and MRN **0.56 → 0.86** from 6 % to 100 % contrast.
- **Choice decoding is high and roughly flat**, and it is **above chance already at 0 % contrast**
  (SCm/MRN ≈ 0.58 with *no stimulus at all*), rising to ~0.95 — because it reflects the *movement
  the mouse made*, not the evidence it saw.
- **The two curves diverge at low contrast and converge at high contrast** (SCm at 6 %: stimulus
  0.36 vs choice 0.81; at 100 %: 0.93 vs 0.95). That gap **is** the answer: choice-decoding
  accuracy is not supposed to track contrast.
- **VISp sits near chance here** for both labels — this single session has only 23 VISp units at
  ≥ 1 Hz and it is the project's weakest region; that is exactly why we replicate across the 5
  triplet sessions rather than trust one. (Note SCm decodes the stimulus *well*: the superior
  colliculus is itself a visual structure, so a strong visual signal there is expected.)

**Bottom line for your colleague.** If a decoder's accuracy "doesn't make sense with coherence",
check *which label* is being decoded. A **choice** decoder should **not** scale with contrast —
the mouse acts on every trial, and motor areas encode the act. A **stimulus** decoder should, and
does. The **window** matters too: an early window (≈ 50–200 ms) isolates the sensory signal; a
broad or peri-movement window is dominated by the action. (Standing project caveat: over
[0, +0.8] s movement has already begun on 92 % of trials, so any late "decision" signal is
entangled with movement — hence the separate `firstMovement`-aligned tensors.)
